# Relaciones aprendibles entre variables (EDA)

Busco relaciones con señal suficiente para un modelo predictivo, dentro de cada tabla y entre tablas. Es un barrido de hipótesis sobre `data/warehouse.duckdb` (solo lectura), complementario a la exploración de `complaints` en `query_silver.ipynb`.

**Método.** Cada hipótesis sigue: proposición → consulta → veredicto. Los veredictos son:

| Veredicto | Significado |
|---|---|
| **señal** | Asociación fuerte y estable; usable como feature o target |
| **estructural** | Relación fuerte pero impuesta por la generación de los datos (p. ej. una variable derivada de otra); reportar, no "descubrir" |
| **transformación** | La columna es una función determinista de otra; no es aprendizaje |
| **fuga** | La columna da la respuesta del target; excluir de features |
| **nulo** | Sin asociación útil (con el n y el efecto a la vista) |

**Advertencia de comparaciones múltiples.** Con ~30 hipótesis, alguna parecerá "señal" por azar. Aquí solo se marca señal lo que tiene efecto grande (decenas de puntos porcentuales o ~determinista) y se mantiene en cortes; los efectos pequeños se reportan como nulos.

**Resumen ejecutivo y uso predictivo:** [`docs/data_findings.md`](../../docs/data_findings.md).

In [ ]:
from pathlib import Path

import duckdb

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
con = duckdb.connect(str(ROOT / "data/warehouse.duckdb"), read_only=True)

## 1. Clientes

### H1. `segment` ~ `credit_score` / ingreso

In [ ]:
con.sql("""
    select segment, count(*) n, min(credit_score) cs_min, max(credit_score) cs_max,
           round(avg(credit_score), 1) cs_avg, round(stddev(credit_score), 1) cs_sd,
           round(avg(estimated_monthly_income), 0) inc_avg
    from customers group by 1 order by cs_avg
""").show(max_width=200)

**Veredicto: estructural.** `segment` es prácticamente una función de `credit_score`: Basic ~600, Student ~649, Plus ~699, Premium ~797 (sd ~40). Predecir segmento con credit_score es aprender el generador, no el negocio.

### H2. `credit_score` ~ ingreso (¿la correlación global se sostiene dentro de segmento?)

In [ ]:
print("Correlación global:")
con.sql("select round(corr(credit_score, estimated_monthly_income), 3) cs_inc_corr from customers").show()

print("Dentro de cada segmento:")
con.sql("""
    select segment, count(*) n,
           round(corr(credit_score, estimated_monthly_income), 3) cs_inc_corr
    from customers group by 1 order by 1
""").show(max_width=200)

**Veredicto: estructural (paradoja de Simpson).** La correlación global es 0.36, pero dentro de cada segmento es ~0. Toda la asociación está *entre* segmentos: ambas variables se generan a partir del segmento. No hay relación credit_score ↔ ingreso que un modelo explote dentro de un segmento.

### H3. `customer_status` (churn) ~ comportamiento (transacciones, quejas, productos, mora, recencia)

In [ ]:
con.sql("""
    with beh as (select customer_id, count(*) n_tx, max(transaction_date) last_tx from transactions group by 1),
    comp as (select customer_id, count(*) n_comp from complaints group by 1),
    prod as (select customer_id, count(*) n_prod, max(days_past_due) max_dpd from products group by 1),
    cci as (select customer_id, count(*) n_call from call_center_interactions group by 1)
    select c.customer_status, count(*) n,
           round(avg(beh.n_tx), 1) avg_tx,
           round(avg(coalesce(comp.n_comp, 0)), 2) avg_comp,
           round(avg(prod.n_prod), 2) avg_prod,
           round(100 * avg((coalesce(prod.max_dpd, 0) > 0)::int), 1) any_dpd_pct,
           round(avg(coalesce(cci.n_call, 0)), 2) avg_call,
           round(avg(datediff('day', beh.last_tx, timestamp '2026-06-17')), 0) avg_recency_d
    from customers c
    left join beh using (customer_id)
    left join comp using (customer_id)
    left join prod using (customer_id)
    left join cci using (customer_id)
    group by 1 order by n desc
""").show(max_width=250)

**Veredicto: nulo.** `customer_status` (Active/Inactive/Suspended/Closed) no distingue nada: 32.6–32.9 transacciones, 0.41–0.45 quejas, ~11.8% con mora, recencia ~43–45 días en todos. No hay modelo de churn posible sobre estas variables.

### H4. Demografía (`gender`, edad, `education_level`, `occupation`) ~ `credit_score`

In [ ]:
con.sql("""
    select gender, count(*) n, round(avg(credit_score), 1) cs
    from customers group by 1 order by 1
""").show(max_width=200)

con.sql("""
    select least(floor(datediff('year', date_of_birth, date '2026-06-17') / 10), 9) age_dec,
           count(*) n, round(avg(credit_score), 1) cs
    from customers group by 1 order by 1
""").show(max_width=200)

con.sql("""
    select education_level, count(*) n, round(avg(credit_score), 1) cs
    from customers group by 1 order by n desc
""").show(max_width=200)

**Veredicto: nulo.** Género, edad y educación dejan el credit_score en ~647 con desviaciones < 2 puntos (n = 150,000).

## 2. Productos

### H5. Morosidad (`days_past_due`) ~ utilización del crédito (`current_balance / credit_limit`)

In [ ]:
con.sql("""
    select product_type,
           least(floor((coalesce(current_balance, 0) / nullif(credit_limit, 0)) / 0.25), 7) util_b,
           count(*) n,
           round(100 * avg((days_past_due > 0)::int), 1) past_due_pct
    from products where credit_limit > 0
    group by all order by 1, 2
""").show(max_rows=40, max_width=200)

**Veredicto: nulo.** La tasa de mora es ~15% en todos los niveles de utilización y todos los productos.

### H6. Morosidad ~ atributos del cliente (ingreso, credit_score)

In [ ]:
con.sql("""
    select least(floor(p.days_past_due / 30), 6) dpd_dec,
           count(*) n,
           round(avg(c.credit_score), 1) cs,
           round(avg(c.estimated_monthly_income), 0) inc
    from products p join customers c using (customer_id)
    group by 1 order by 1
""").show(max_width=200)

**Veredicto: nulo.** Mismo credit_score (~647) e ingreso en todos los niveles de mora. Consistente con `docs/known_issues.md` (mora independiente del credit_score): no hay señal para un modelo de riesgo crediticio.

### H7. `interest_rate` / `credit_limit` ~ `product_type`

In [ ]:
con.sql("""
    select product_type, count(*) n,
           round(avg(interest_rate), 2) ir_avg,
           round(avg(credit_limit), 0) lim_avg
    from products group by 1 order by n desc
""").show(max_width=200)

**Veredicto: estructural.** `interest_rate` es constante por tipo de producto (tarjeta 31.5, personal 20.03, hipoteca 8.99, ahorro 2.25…). No hay variación que aprender.

## 3. Transacciones

### H8-H9. Fraude ~ monto y hora del día

In [ ]:
con.sql("""
    select least(floor(abs(amount) / 250), 8) amt_b, count(*) n,
           round(100 * avg(is_fraud::int), 3) fraud_pct
    from transactions group by 1 order by 1
""").show(max_width=200)

con.sql("""
    select least(floor(hour(transaction_date) / 3), 8) tod_b, count(*) n,
           round(100 * avg(is_fraud::int), 3) fraud_pct
    from transactions group by 1 order by 1
""").show(max_width=200)

**Veredicto: nulo.** Fraude plano ~0.1% por franja de monto y por franja horaria (n = 4.4M). Recordatorio: `fraud_score` da `is_fraud` directamente (`docs/known_issues.md`), así que un "modelo de fraude" sobre esa columna tampoco cuenta.

### H10-H11. Tasa de rechazo y fraude ~ canal / país / categoría de comercio

In [ ]:
con.sql("""
    select channel, transaction_country, count(*) n,
           round(100 * avg((transaction_status = 'Declined')::int), 1) declined_pct
    from transactions group by all order by n desc limit 12
""").show(max_width=200)

con.sql("""
    select merchant_category, channel, count(*) n,
           round(100 * avg(is_fraud::int), 3) fraud_pct
    from transactions
    group by all having count(*) > 5000
    order by fraud_pct desc limit 10
""").show(max_width=200)

**Veredicto: nulo.** Rechazo ~5% en todas las combinaciones de canal y país; fraude 0.08–0.14% por categoría de comercio × canal. La uniformidad de los códigos de rechazo ya está documentada; aquí se confirma que no hay patrón geográfico ni de comercio.

## 4. Interacciones de call center

### H12. FCR (`was_resolved`) ~ motivo de contacto (`contact_reason`)

In [ ]:
con.sql("""
    select contact_reason, count(*) n,
           round(100 * avg(was_resolved::int), 1) resolved_pct,
           round(100 * avg(requires_followup::int), 1) followup_pct
    from call_center_interactions
    group by 1 order by resolved_pct
""").show(max_width=200)

**Veredicto: señal.** 43.6% (Queja) → 91.5% (Transaccional), n = 686,296. Es la relación más fuerte del barrido y sí es usable: `contact_reason` se conoce al inicio de la llamada. Un modelo de "probabilidad de resolución en primera llamada" tendría su base aquí.

### H13. FCR ~ duración de la llamada

In [ ]:
con.sql("""
    select least(floor(duration_seconds / 120), 8) dur_b, count(*) n,
           round(100 * avg(was_resolved::int), 1) resolved_pct
    from call_center_interactions group by 1 order by 1
""").show(max_width=200)

**Veredicto: señal parcial, pero post-hoc y confundida.** La resolución cae de 89.7% (0–2 min) a 57% (8–10 min) y vuelve a 76.7% en el grupo ≥16 min (distribución bimodal de duración). La duración solo se conoce *después* de la llamada: sirve para monitoreo, no para predicción al inicio. Parte del efecto es composición (las llamadas de Queja son las más largas, 435 s de media, y las menos resueltas).

### H14. FCR ~ sentimiento detectado

In [ ]:
con.sql("""
    select detected_sentiment, count(*) n,
           round(100 * avg(was_resolved::int), 1) resolved_pct
    from call_center_interactions group by 1 order by n desc
""").show(max_width=200)

print("Dentro del motivo (Queja y Transaccional):")
con.sql("""
    select reason_category, detected_sentiment, count(*) n,
           round(100 * avg(was_resolved::int), 1) resolved_pct
    from call_center_interactions
    where reason_category in ('Queja', 'Transaccional')
    group by all order by 1, 2
""").show(max_width=250)

**Veredicto: nulo (confundido).** Globalmente "Neutral" resuelve 82.6% vs ~64.5% el resto, pero dentro de cada motivo el sentimiento no mueve nada (Queja: 43.2–44.0% en todos los sentimientos). El efecto global es composición: **todas** las llamadas Transaccionales (240,056) son "Neutral". El sentimiento no aporta más allá del motivo.

### H15. FCR ~ agente (`experience_level`, `specialty`)

In [ ]:
con.sql("""
    select a.experience_level, a.specialty, count(*) n,
           round(100 * avg(i.was_resolved::int), 1) resolved_pct
    from call_center_interactions i
    join service_agents a on a.agent_id = i.agent_id
    group by all order by n desc limit 15
""").show(max_width=250)

**Veredicto: nulo.** 76.2–77.2% de resolución en todos los niveles de experiencia y especialidades. Asignar al "agente correcto" no se puede justificar con los datos.

### H16. `requires_followup` ~ `was_resolved` (¿columna derivada?)

In [ ]:
con.sql("""
    select was_resolved, requires_followup, count(*) n
    from call_center_interactions group by all order by all
""").show(max_width=200)

**Veredicto: transformación parcial.** No existe la combinación (no resuelta, sin follow-up): `requires_followup = false` implica `was_resolved = true`. Usar ambas columnas como targets separados es redundante en una dirección.

## 5. Encuestas de satisfacción

### H17. `main_score` ~ resultado de la interacción (`was_resolved`), por tipo de encuesta

In [ ]:
con.sql("""
    select s.survey_type, i.was_resolved, count(*) n,
           round(avg(s.main_score), 2) score_avg,
           min(s.main_score) score_min, max(s.main_score) score_max
    from satisfaction_surveys s
    join call_center_interactions i using (interaction_id)
    group by all order by 1, 2
""").show(max_width=200)

print("Distribución completa:")
con.sql("""
    select s.survey_type, i.was_resolved, s.main_score, count(*) n
    from satisfaction_surveys s
    join call_center_interactions i using (interaction_id)
    group by all order by 1, 2, 3
""").show(max_rows=30, max_width=200)

**Veredicto: señal fuerte (casi determinista).** Resuelta vs no resuelta: CSAT/CES 3.0 vs 2.0, NPS 6.0 vs 3.0 (n = 212,759). La distribución se desplaza ±1–3 puntos de forma consistente. Es el único target de satisfacción con señal, pero sale *después* de la llamada: un "predictor de satisfacción" solo aprendería `survey_type` + `was_resolved`.

### H18. `main_score` ~ respuestas de las preguntas del cuestionario

In [ ]:
con.sql("""
    select survey_type, count(*) n,
           round(avg(main_score), 2) score_avg,
           round(avg(question_1_response), 2) q1,
           round(avg(question_2_response), 2) q2,
           round(avg(question_3_response), 2) q3
    from satisfaction_surveys group by 1 order by n desc
""").show(max_width=200)

con.sql("""
    select round(corr(main_score, question_1_response), 3) q1_corr,
           round(corr(main_score, question_2_response), 3) q2_corr,
           round(corr(main_score, question_3_response), 3) q3_corr
    from satisfaction_surveys
""").show()

**Veredicto: nulo.** Las preguntas 1–3 valen ~3.0 siempre y no correlacionan con el score (r ≤ 0.01). No hay sub-señal de satisfacción en el cuestionario.

### H19. `comment_sentiment` ~ `main_score` (por tipo de encuesta)

In [ ]:
con.sql("""
    select survey_type, comment_sentiment, count(*) n,
           round(avg(main_score), 2) score_avg
    from satisfaction_surveys group by all order by 1, 2
""").show(max_width=200)

**Veredicto: señal parcial (plantillas).** Negative 2.5–2.6 vs Positive 4.0, pero los comentarios son 13 plantillas (`docs/known_issues.md`), así que el sentimiento es una búsqueda en tabla del texto, no una opinión libre. Útil para demostrar pipeline de texto; no como aprendizaje.

### H20. `nps_category` ~ `main_score` (chequeo de transformación)

In [ ]:
con.sql("""
    select nps_category, count(*) n, min(main_score) mn, max(main_score) mx
    from satisfaction_surveys group by 1 order by 1
""").show(max_width=200)

**Veredicto: transformación.** `nps_category` es función determinista de `main_score` (Detractor 2–6, Passive 7, sin Promoters — escalas truncadas). No es aprendizaje.

## 6. Campañas de marketing

### H21. Embudo (entrega → apertura → clic → conversión) ~ canal

In [ ]:
con.sql("""
    select send_channel, count(*) n,
           round(100 * avg(was_delivered::int), 2) delivered_pct,
           round(100 * avg(was_opened::int), 2) opened_pct,
           round(100 * avg(was_clicked::int), 2) clicked_pct,
           round(100 * avg(had_conversion::int), 3) converted_pct
    from campaign_sends group by 1 order by n desc
""").show(max_width=200)

print("Tasas condicionales por etapa:")
con.sql("""
    select send_channel,
           round(100 * avg(was_clicked::int / nullif(was_opened::int, 0)), 2) click_given_open_pct,
           round(100 * avg(had_conversion::int / nullif(was_clicked::int, 0)), 2) conv_given_click_pct
    from campaign_sends group by 1 order by 1
""").show(max_width=200)

**Veredicto: señal (solo en la apertura).** SMS abre 49.9% vs Email 30.0% vs Push 40.1% (n = 1.75M); WhatsApp y Voice no registran apertura. Pero las tasas condicionales son constantes: clic|apertura ~20% y conversión|clic ~10% en todos los canales. El canal solo cambia la primera etapa; el embudo posterior es uniforme.

### H22. Variación entre plantillas dentro de una misma campaña (¿efecto plantilla o ruido?)

In [ ]:
con.sql("""
    with t as (
        select campaign_id, template_used, count(*) n,
               round(100 * avg(was_opened::int), 2) opened_pct,
               round(100 * avg(had_conversion::int), 3) conv_pct
        from campaign_sends where template_used is not null
        group by all having count(*) > 500
    )
    select campaign_id, count(*) n_templates,
           round(min(opened_pct), 2) open_min, round(max(opened_pct), 2) open_max,
           round(min(conv_pct), 3) conv_min, round(max(conv_pct), 3) conv_max
    from t group by 1 order by 1 limit 12
""").show(max_width=250)

print("Posición de plantilla agregada sobre todas las campañas:")
con.sql("""
    select substr(template_used, length(template_used), 1) template_no, count(*) n,
           round(100 * avg(was_opened::int), 2) opened_pct,
           round(100 * avg(had_conversion::int), 3) conv_pct
    from campaign_sends where template_used is not null
    group by 1 order by 1
""").show(max_width=200)

**Veredicto: nulo (ruido).** Dentro de una campaña las 5 plantillas difieren en apertura por 1–3 puntos y en conversión por hasta 4x (0.22% vs 0.89%), pero con n ≈ 900 por plantilla y ~5 conversiones por celda. Agregando la posición de plantilla sobre todas las campañas: 0.55–0.58% de conversión, plano. No hay efecto plantilla real.

### H23. Engagement ~ `accepts_marketing` del cliente

In [ ]:
con.sql("""
    select c.accepts_marketing, count(*) n,
           round(100 * avg(s.was_opened::int), 1) opened_pct,
           round(100 * avg(s.had_conversion::int), 3) converted_pct
    from campaign_sends s join customers c using (customer_id)
    group by 1 order by 1
""").show(max_width=200)

**Veredicto: nulo (y sorpresa de calidad).** Clientes que **no** aceptan marketing abren y convierten igual (38.6% vs 38.6%; 0.57% vs 0.55%). El consentimiento no filtra los envíos ni el engagement. Es un hallazgo de calidad de datos más que de modelado.

## 7. Transcripts: chequeos de fuga para el router

### H24. `customer_text` ~ `contact_reason` (¿se puede entrenar un router con el texto real?)

In [ ]:
con.sql("""
    with x as (
        select t.customer_text, i.contact_reason, count(*) n
        from call_transcripts t
        join call_center_interactions i using (interaction_id)
        group by 1, 2
    ), y as (
        select customer_text, sum(n) tot, max(n) mx
        from x group by 1
    )
    select count(*) n_texts, sum(tot) n_rows,
           round(100 * sum(mx) / sum(tot), 1) majority_purity_pct
    from y
""").show(max_width=200)

con.sql("""
    select count(*) n,
           count(distinct customer_text) distinct_customer_text,
           count(distinct full_text) distinct_full_text
    from call_transcripts
""").show(max_width=200)

**Veredicto: nulo.** 42 textos de cliente distintos (dos frases base de consulta de saldo + combinaciones de 3 respuestas de seguimiento) para 171,321 filas. La pureza de la clase mayoritaria por texto es 34.9% ≈ la base de la clase mayoritaria: el texto no predice el motivo. Un router no se puede entrenar con el texto real; hace falta texto generado y etiquetado (como ya asume `ml/README.md`).

### H25. `main_topics` / `detected_intents` ~ `contact_reason` (chequeo de fuga)

In [ ]:
con.sql("""
    select count(*) n,
           round(100 * avg((t.main_topics = i.contact_reason)::int), 2) exact_match_pct
    from call_transcripts t
    join call_center_interactions i using (interaction_id)
    where t.main_topics is not null
""").show(max_width=200)

con.sql("""
    select count(distinct detected_intents) n_intents,
           count(distinct main_topics) n_topics
    from call_transcripts
""").show(max_width=200)

**Veredicto: fuga.** `main_topics` = `contact_reason` en el 100% de las filas (n = 171,321). `detected_intents` es constante (`consulta_general`). Si el router se entrena con texto, estas columnas no pueden entrar como features: dan la etiqueta. Es exactamente el chequeo de fuga que pedía `ml/README.md`.

## 8. Entre tablas

### H26-H27. ¿La llamada precede a la queja? (unión temporal; `origin_interaction_id` es siempre null)

In [ ]:
print("Quejas con una llamada del mismo cliente en los 7 días anteriores:")
con.sql("""
    select co.category, count(*) n_complaints,
           round(100 * avg(exists(
               select 1 from call_center_interactions i
               where i.customer_id = co.customer_id
                 and i.interaction_date between co.creation_date - interval 7 day and co.creation_date
           )::int), 1) had_call_before_pct
    from complaints co group by 1 order by n_complaints desc limit 10
""").show(max_width=250)

print("Llamadas seguidas de una queja en 7 días, por motivo:")
con.sql("""
    select c.reason_category, count(*) n_calls,
           round(100 * avg(exists(
               select 1 from complaints co
               where co.customer_id = c.customer_id
                 and co.creation_date between c.interaction_date and c.interaction_date + interval 7 day
           )::int), 2) complained_within_7d_pct
    from call_center_interactions c group by 1 order by n_calls desc
""").show(max_width=250)

**Veredicto: nulo.** Solo ~3% de las quejas tienen una llamada previa y solo ~0.3% de las llamadas se siguen de una queja, sin variación por motivo ni categoría. Confirma que el flujo llamada → queja no existe como relación utilizable.

### H28. Coincidencia de acento cliente–agente ~ resultado de la llamada

In [ ]:
con.sql("""
    select (i.customer_detected_accent = a.native_accent) accent_match, count(*) n,
           round(100 * avg(i.was_resolved::int), 1) resolved_pct
    from call_center_interactions i
    join service_agents a on a.agent_id = i.agent_id
    group by 1 order by 1
""").show(max_width=200)

**Veredicto: nulo.** 76.7% vs 76.7% (n = 481,546 con acento conocido). El emparejamiento de acento no importa.

### H29. Envío de campaña ~ transacción posterior en 7 días

In [ ]:
con.sql("""
    select s.send_channel, count(*) n,
           round(100 * avg(exists(
               select 1 from transactions t
               where t.customer_id = s.customer_id
                 and t.transaction_date between s.send_date and s.send_date + interval 7 day
           )::int), 1) tx_within_7d_pct
    from campaign_sends s group by 1 order by n desc
""").show(max_width=200)

**Veredicto: nulo.** 16.4–16.6% en todos los canales (n = 1.75M). La campaña no mueve la actividad transaccional en los datos.

### H30. Historial de quejas del cliente ~ score de satisfacción

In [ ]:
con.sql("""
    with agg as (select customer_id, count(*) n_comp from complaints group by 1)
    select s.survey_type, least(coalesce(agg.n_comp, 0), 4) n_complaints,
           count(*) n, round(avg(s.main_score), 2) score_avg
    from satisfaction_surveys s
    left join agg using (customer_id)
    group by 1, 2 order by 1, 2
""").show(max_width=200)

**Veredicto: nulo.** Score plano (2.77 CES/CSAT; 5.30 NPS) sin importar cuántas quejas tenga el cliente.

### H31. `service_agents.avg_csat` ~ satisfacción real medida en sus interacciones

In [ ]:
con.sql("""
    with real as (
        select agent_id, avg(main_score) realized, count(*) n_surveys
        from satisfaction_surveys group by 1 having count(*) >= 30
    )
    select count(*) n_agents,
           round(corr(a.avg_csat, r.realized), 3) csat_realized_corr
    from service_agents a join real r using (agent_id)
""").show()

**Veredicto: nulo (y hallazgo de calidad).** r = −0.03 entre el `avg_csat` declarado del agente y la satisfacción real medida (1,090 agentes con ≥30 encuestas). El atributo del agente no refleja su desempeño observado.

## 9. Profundización en los tres hallazgos con señal

### D1. FCR por motivo, dentro de duración y dentro de sentimiento (¿confundido?)

In [ ]:
con.sql("""
    select reason_category, least(floor(duration_seconds / 180), 5) dur_b, count(*) n,
           round(100 * avg(was_resolved::int), 1) resolved_pct
    from call_center_interactions
    group by all order by 1, 2
""").show(max_rows=40, max_width=250)

con.sql("""
    select reason_category, detected_sentiment, count(*) n,
           round(100 * avg(was_resolved::int), 1) resolved_pct
    from call_center_interactions
    group by all order by 1, 2
""").show(max_rows=40, max_width=250)

**Veredicto: señal robusta.** El efecto del motivo se mantiene dentro de cada franja de duración (Queja ~43% vs Transaccional ~75–94%) y dentro de cada sentimiento. El motivo es el predictor; duración y sentimiento no añaden.

### D2. Satisfacción dada la resolución: ¿ruido o desplazamiento consistente?

In [ ]:
con.sql("""
    with counts as (
        select s.survey_type, i.was_resolved, s.main_score, count(*) n
        from satisfaction_surveys s
        join call_center_interactions i using (interaction_id)
        group by 1, 2, 3
    )
    select *, round(100.0 * n / sum(n) over (partition by survey_type, was_resolved), 1) pct
    from counts order by 1, 2, 3
""").show(max_rows=30, max_width=250)

**Veredicto: señal consistente.** Cada tipo de encuesta desplaza la distribución del score alrededor de un punto central fijo según la resolución (CSAT/CES: 2 ↔ 3; NPS: 3 ↔ 6). Es aprendible, pero la regla es casi determinista.

### D3. Embudo de campaña: ¿dónde actúa realmente el canal?

In [ ]:
con.sql("""
    select send_channel,
           count(*) n,
           round(100 * avg(was_delivered::int), 2) delivered_pct,
           round(100 * avg(was_opened::int), 2) opened_pct,
           round(100 * avg(was_opened::int / nullif(was_delivered::int, 0)), 2) open_given_delivered_pct
    from campaign_sends group by 1 order by n desc
""").show(max_width=250)

**Veredicto: señal acotada.** La entrega es ~94% en todos los canales; toda la diferencia del canal está en la apertura (Email 30% → SMS 50%). Las etapas posteriores son constantes.

## 10. Expansión de las señales

### E1. FCR ~ motivo: estabilidad (país, año, mes, canal, antigüedad)

In [ ]:
print("Por país:")
con.sql("""
    select c.country, i.contact_reason, count(*) n,
           round(100 * avg(i.was_resolved::int), 1) resolved_pct
    from call_center_interactions i join customers c using (customer_id)
    group by all order by 1, 2
""").show(max_rows=20, max_width=250)

print("Por año:")
con.sql("""
    select year(i.interaction_date) yr, i.contact_reason, count(*) n,
           round(100 * avg(i.was_resolved::int), 1) resolved_pct
    from call_center_interactions i
    group by all order by 1, 2
""").show(max_rows=30, max_width=250)

print("Por antigüedad del cliente:")
con.sql("""
    with cust as (
        select c.customer_id,
               datediff('day', c.registration_date, i.interaction_date) tenure_days,
               i.contact_reason, i.was_resolved
        from call_center_interactions i join customers c using (customer_id)
    )
    select contact_reason,
           round(100 * avg(was_resolved::int) filter (where tenure_days < 365), 1) lt_1y,
           round(100 * avg(was_resolved::int) filter (where tenure_days between 365 and 1095), 1) y1_3,
           round(100 * avg(was_resolved::int) filter (where tenure_days > 1095), 1) gt_3y,
           count(*) n
    from cust group by 1 order by n desc
""").show(max_width=250)

print("Por canal de interacción:")
con.sql("""
    select channel, interaction_type, count(*) n,
           round(100 * avg(was_resolved::int), 1) resolved_pct
    from call_center_interactions
    group by all order by n desc limit 10
""").show(max_width=250)

**Veredicto: señal estable.** Las tasas por motivo se mantienen dentro de ±0.5 puntos entre Argentina/Colombia/México, entre 2023–2026, entre meses y entre antigüedades. El canal no cambia nada (76.5–77.8% en todos). La señal es el motivo; no hay deriva ni interacción que capturar.

### E2. FCR: cuánto explica el motivo (entropía) y contra qué baseline compite

In [ ]:
print("Entropía de was_resolved (bits):")
con.sql("""
    with p as (select was_resolved, count(*) n from call_center_interactions group by 1),
    tot as (select sum(n) n from p)
    select round(-sum((n / (select n from tot)) * log2(n / (select n from tot))), 4) entropy_overall_bits
    from p
""").show()

con.sql("""
    with p as (select contact_reason, was_resolved, count(*) n
               from call_center_interactions group by 1, 2),
    g as (select contact_reason, sum(n) gn from p group by 1),
    e as (select p.contact_reason, -sum((p.n / g.gn) * log2(p.n / g.gn)) h
          from p join g using (contact_reason) group by 1, g.gn)
    select round(sum(g.gn * e.h) / sum(g.gn), 4) cond_entropy_given_reason_bits
    from e join g using (contact_reason)
""").show()

print("Accuracy de las dos reglas triviales:")
con.sql("""
    with per_reason as (
        select contact_reason, was_resolved, count(*) n,
               row_number() over (partition by contact_reason order by count(*) desc) rn
        from call_center_interactions group by 1, 2
    )
    select round(100 * sum(n) filter (where rn = 1) / sum(n), 2) per_reason_majority_acc_pct,
           sum(n) total
    from per_reason
""").show()
con.sql("""
    select round(100 * avg(was_resolved::int), 2) always_resolved_acc_pct
    from call_center_interactions
""").show()

**Veredicto.** Saber el motivo baja la incertidumbre de 0.784 a 0.657 bits (−16%). La regla de mayoría por motivo da 78.8% de accuracy vs 76.7% de "siempre resolver": +2.2 puntos, modesto en accuracy pero la señal es de *calibración* (riesgo), no de clasificación dura.

### E3. FCR: framing accionable — ¿"no se resolverá en primera llamada"?

In [ ]:
con.sql("""
    with pred as (
        select
            sum((not was_resolved and contact_reason = 'Queja')::int) tp,
            sum((was_resolved and contact_reason = 'Queja')::int) fp,
            sum((not was_resolved and contact_reason <> 'Queja')::int) fn,
            sum((was_resolved and contact_reason <> 'Queja')::int) tn
        from call_center_interactions
    )
    select tp, fp, fn, tn,
           round(100.0 * tp / (tp + fp), 1) precision_pct,
           round(100.0 * tp / (tp + fn), 1) recall_pct,
           round(100.0 * (tp + fn) / (tp + fp + fn + tn), 1) base_rate_pct
    from pred
""").show(max_width=250)

print("Tabla de calibración (probabilidad por motivo de NO resolver):")
con.sql("""
    select contact_reason, count(*) n,
           round(100 * avg((not was_resolved)::int), 1) not_resolved_pct
    from call_center_interactions group by 1 order by not_resolved_pct desc
""").show(max_width=250)

**Veredicto.** Marcando solo `Queja` como "riesgo de no resolución": precisión 56.4%, recall 41.2%, contra una base del 23.4% — **2.4× lift**. Es el framing para el sistema: al clasificar el motivo (lo que ya hace el router), el agente conoce el riesgo y puede ofrecer follow-up o escalar temprano. La tabla de calibración es la "probabilidad" que entrega el modelo.

### E4. Satisfacción: exactitud del predictor y varianza residual

In [ ]:
print("Accuracy del predictor moda-por-celda vs moda global:")
con.sql("""
    with cells as (
        select s.survey_type, i.was_resolved, s.main_score, count(*) n,
               row_number() over (partition by s.survey_type, i.was_resolved order by count(*) desc) rn
        from satisfaction_surveys s
        join call_center_interactions i using (interaction_id)
        group by 1, 2, 3
    )
    select round(100 * sum(n) filter (where rn = 1) / sum(n), 2) cell_mode_acc_pct, sum(n) total
    from cells
""").show()
con.sql("""
    with gm as (select main_score from satisfaction_surveys group by 1 order by count(*) desc limit 1)
    select round(100 * avg((main_score = (select main_score from gm))::int), 2) global_mode_acc_pct
    from satisfaction_surveys
""").show()

print("Desviación estándar residual dentro de cada celda:")
con.sql("""
    select s.survey_type, i.was_resolved, count(*) n,
           round(stddev(s.main_score), 3) sd, round(avg(s.main_score), 2) mean
    from satisfaction_surveys s
    join call_center_interactions i using (interaction_id)
    group by 1, 2 order by 1, 2
""").show(max_width=250)

print("¿Añade algo el motivo dentro de la celda?")
con.sql("""
    select s.survey_type, i.contact_reason, i.was_resolved, count(*) n,
           round(avg(s.main_score), 2) score
    from satisfaction_surveys s
    join call_center_interactions i using (interaction_id)
    group by all order by 1, 2, 3
""").show(max_rows=40, max_width=250)

**Veredicto.** La moda por celda (`survey_type` × `was_resolved`) acierta 59.3% vs 42.5% de la moda global (+16.8 puntos). El residuo dentro de la celda es ruido pequeño y homogéneo (sd 0.54 en CSAT/CES, 0.82 en NPS), y ni el motivo, ni el sentimiento, ni el agente, ni el segmento lo mueven. El modelo completo de satisfacción es esa celda de 6 entradas.

### E5. Campañas: la varianza entre campañas es ruido y el embudo cierra exacto

In [ ]:
print("Dispersión de tasas de apertura entre campañas del mismo canal (Email, n > 1000):")
con.sql("""
    with x as (
        select campaign_id, round(100 * avg(was_opened::int), 2) open_pct, count(*) n
        from campaign_sends where send_channel = 'Email' and was_delivered
        group by 1 having count(*) > 1000
    )
    select count(*) n_campaigns, round(avg(open_pct), 2) mean_open_pct,
           round(stddev(open_pct), 2) sd_open_pct,
           round(min(open_pct), 2) mn, round(max(open_pct), 2) mx
    from x
""").show(max_width=250)

print("Aritmética del embudo: entregado × apertura × 0.20 × 0.10 vs conversiones reales por 1000:")
con.sql("""
    select send_channel, count(*) n,
           round(1000 * avg(was_delivered::int)
                 * avg(was_opened::int) filter (where was_opened is not null)
                 * 0.20 * 0.10, 2) est_conversions_per_1000,
           round(1000 * avg(had_conversion::int), 2) actual_conversions_per_1000
    from campaign_sends group by 1 order by n desc
""").show(max_width=250)

print("Chequeo de fuga: open_device / click_date / conversion_value solo existen tras el evento:")
con.sql("""
    select was_opened, (open_device is not null) has_device, count(*) n
    from campaign_sends group by all order by all
""").show(max_width=200)
con.sql("""
    select had_conversion, (conversion_value > 0) has_value, count(*) n
    from campaign_sends group by all order by all
""").show(max_width=200)

**Veredicto.** Entre campañas del mismo canal la apertura varía 27.9–31.7% (sd 0.62): el canal *es* el modelo, la campaña no. El embudo cierra exacto (estimado 5.64 vs real 5.64 en Email; 9.39 vs 9.37 en SMS): las tasas condicionales son constantes por generación. `open_device`, `open_country`, `click_date` y `conversion_value` solo se llenan cuando el evento ocurre — son **fuga**, no features.

## 11. `digital_events`: la tabla que faltaba (15.6M filas, solo bronce)

No está en silver. Se lee directo de `data/bronze/data/digital_events/**/*.csv` (ver nota en `docs/known_issues.md`: cargar solo las columnas necesarias).

In [ ]:
GLOB = str(ROOT / 'data/bronze/data/digital_events/**/*.csv')

con.sql(f"""
    select event_type, event_category, count(*) n
    from read_csv('{GLOB}', header=true, union_by_name=true)
    group by all order by n desc limit 12
""").show(max_width=250)

print("Engagement digital ~ churn:")
con.sql(f"""
    with eng as (
        select customer_id, count(*) n_events, count(distinct session_id) n_sessions
        from read_csv('{GLOB}', header=true, union_by_name=true)
        where customer_id is not null
        group by 1
    )
    select c.customer_status, count(*) n,
           round(avg(eng.n_events), 1) avg_events,
           round(avg(eng.n_sessions), 1) avg_sessions
    from customers c join eng using (customer_id)
    group by 1 order by n desc
""").show(max_width=250)

print("Tasa de Error/Purchase por segmento:")
con.sql(f"""
    with ev as (
        select customer_id,
               avg((event_type = 'Error')::int) err_rate,
               avg((event_type = 'Purchase')::int) purchase_rate
        from read_csv('{GLOB}', header=true, union_by_name=true)
        where customer_id is not null
        group by 1
    )
    select c.segment, count(*) n,
           round(avg(ev.err_rate), 4) err_rate,
           round(avg(ev.purchase_rate), 4) purchase_rate
    from customers c join ev using (customer_id)
    group by 1 order by n desc
""").show(max_width=250)

print("¿Un evento Error precede a una queja en 3 días?")
con.sql(f"""
    with err as (
        select distinct customer_id, event_date
        from read_csv('{GLOB}', header=true, union_by_name=true)
        where event_type = 'Error' and customer_id is not null
    )
    select count(*) n_err_events,
           round(100 * avg(exists(
               select 1 from complaints co
               where co.customer_id = e.customer_id
                 and co.creation_date between e.event_date and e.event_date + interval 3 day
           )::int), 2) complained_within_3d_pct
    from err e
""").show(max_width=250)

print("Duración media por plataforma y UTM ~ compra:")
con.sql(f"""
    select platform, is_mobile, count(*) n, round(avg(duration_seconds), 1) avg_dur
    from read_csv('{GLOB}', header=true, union_by_name=true)
    group by all order by n desc limit 8
""").show(max_width=250)
con.sql(f"""
    select (utm_source is not null) has_utm, count(*) n,
           round(100 * avg((event_type = 'Purchase')::int), 3) purchase_pct,
           round(100 * avg((event_type = 'Error')::int), 3) error_pct
    from read_csv('{GLOB}', header=true, union_by_name=true)
    group by 1 order by 1
""").show(max_width=250)

**Veredicto: nulo en todo.** El engagement digital (79.2 eventos, 9.8 sesiones de media) no distingue el estado del cliente; las tasas de Error/Purchase son idénticas por segmento (2.28% / 1.52%); un Error precede a una queja en el 0.12% de los casos; la duración es 152.5 s en todas las plataformas; y las filas con UTM **nunca** tienen Purchase ni Error (0.000%): los parámetros UTM son decorativos. Añade dos problemas de calidad a `docs/known_issues.md`.

## 12. Resumen

| # | Relación | Veredicto | Evidencia (n, efecto) | Uso predictivo |
|---|---|---|---|---|
| H1 | `segment` ~ `credit_score`/ingreso | estructural | 150k; bandas 600/649/699/797 ± 40 | Ninguno (derivado del generador) |
| H2 | `credit_score` ↔ ingreso | estructural (Simpson) | r=0.36 global, ~0 por segmento | Ninguno dentro de segmento |
| H3 | `customer_status` ~ comportamiento | nulo | 150k; diferencias < 3% en todo | No hay modelo de churn |
| H4 | demografía ~ `credit_score` | nulo | 150k; Δ < 2 puntos | — |
| H5 | mora ~ utilización | nulo | 400k; ~15% en todas las franjas | — |
| H6 | mora ~ ingreso/score | nulo | 400k; planos | No hay modelo de riesgo |
| H7 | `interest_rate` ~ tipo de producto | estructural | 400k; constante por tipo | — |
| H8-H9 | fraude ~ monto/hora | nulo | 4.4M; ~0.1% plano | — |
| H10-H11 | rechazo/fraude ~ canal/país/comercio | nulo | 4.4M; 5% / 0.1% plano | — |
| **H12** | **FCR ~ `contact_reason`** | **señal** | **686k; 43.6% → 91.5%; estable en país/año/mes/canal; entropía 0.78→0.66 bits; flag Queja 56.4% precisión (2.4× lift)** | **Target/feature viable (router, triage, riesgo)** |
| H13 | FCR ~ duración | señal parcial (post-hoc) | 686k; 90% → 57% | Monitoreo, no predicción al inicio |
| H14 | FCR ~ sentimiento | nulo (confundido) | 686k; efecto global desaparece por motivo | — |
| H15 | FCR ~ agente | nulo | 686k; 76.2–77.2% | — |
| H16 | `requires_followup` ~ `was_resolved` | transformación parcial | 686k; (false,false) no existe | Redundante |
| **H17** | **`main_score` ~ `was_resolved` × tipo** | **señal (casi determinista)** | **213k; 3 vs 2 (CSAT/CES), 6 vs 3 (NPS); moda-celda 59.3% vs 42.5%; residuo sd 0.54–0.82** | **Target viable, pero solo post-llamada** |
| H18 | `main_score` ~ preguntas 1–3 | nulo | 213k; r ≤ 0.01 | — |
| H19 | `main_score` ~ `comment_sentiment` | señal parcial (plantillas) | 213k; 2.5 vs 4.0 | Demo de texto, no aprendizaje |
| H20 | `nps_category` ~ `main_score` | transformación | 213k; determinista | — |
| **H21** | **embudo ~ canal** | **señal (solo apertura)** | **1.75M; SMS 50% vs Email 30% aperturas; sd entre campañas 0.62; embudo cierra exacto** | **Priorización de canal acotada** |
| J1-J7 | `digital_events` (15.6M) ~ todo | nulo | engagement, errores, UTM, duración: planos; UTM nunca convierte | — |
| H22 | plantilla ~ conversión | nulo (ruido) | 1.75M; posición de plantilla plana 0.55–0.58% | — |
| H23 | engagement ~ `accepts_marketing` | nulo (hallazgo de calidad) | 1.75M; 38.6% = 38.6% | — |
| H24 | `customer_text` ~ `contact_reason` | nulo | 171k; 42 textos, pureza 34.9% | Router requiere texto generado |
| H25 | `main_topics` ~ `contact_reason` | **fuga** | 171k; 100% igual | Excluir de features |
| H26-H27 | llamada ↔ queja | nulo | 686k / 67k; ~3% y ~0.3% | — |
| H28 | acento coincide ~ resultado | nulo | 482k; 76.7% = 76.7% | — |
| H29 | campaña ~ transacción posterior | nulo | 1.75M; 16.5% plano | — |
| H30 | historial de quejas ~ score | nulo | 213k; plano | — |
| H31 | `avg_csat` ~ satisfacción real | nulo (calidad) | 1,090 agentes; r = −0.03 | — |

**Conclusión para modelado.** De ~35 hipótesis, tres tienen señal utilizable y ahora cuantificada contra baseline: FCR ~ motivo (estable en todos los cortes; el flag "Queja" da 2.4× lift sobre la base para anticipar no-resolución; entropía −16%), satisfacción ~ resolución (moda-celda +16.8 puntos sobre global, sin residual aprendible) y apertura ~ canal (el embudo cierra exacto; entre campañas solo hay ruido). Todo lo demás —incluidos los 15.6M de `digital_events`— es plano o estructural. La conclusión de `docs/disputes_findings.md` se mantiene: para el flujo de disputas, el histórico no aporta targets a nivel de caso; el componente aprendido viable es el router con texto generado, cuyo motivo clasificado alimenta además el riesgo de FCR.